In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
DATA = Path("../data/amazon_computers.pt")
OUT = Path("../outputs/")

ck = torch.load(DATA, weights_only=False)
g, feats, labels = ck["graph"], ck["feats"], ck["labels"]
idx_tr, idx_va, idx_te = ck["idx_train"], ck["idx_val"], ck["idx_test"]

In [3]:
def accuracy(logits, y):
    return (logits.argmax(1) == y).float().mean().item()

t = torch.from_numpy(np.load(OUT / "teacher_logits.npz")["logits"])
s = torch.from_numpy(np.load(OUT / "student_logits_mixup.npz")["logits"])
t_acc = accuracy(t[idx_te], labels[idx_te])
s_acc = accuracy(s[idx_te], labels[idx_te])
print(f"teacher {t_acc:.4f}  mixup {s_acc:.4f}")

teacher 0.8232  mixup 0.8200


In [4]:
van_path = OUT / "vanilla_logits.npz"

if van_path.exists():
    van = torch.from_numpy(np.load(van_path)["logits"])
    v_acc = accuracy(van[idx_te], labels[idx_te])
    print(f"loaded vanilla {v_acc:.4f}")
else:
    print("train vanilla 150ep")
    d, C = feats.shape[1], int(labels.max() + 1)

    class MLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(d, 512)
            self.fc2 = nn.Linear(512, C)
            self.drop = nn.Dropout(0.5)

        def forward(self, x):
            return self.fc2(self.drop(F.relu(self.fc1(x))))

    m = MLP()
    opt = torch.optim.Adam(m.parameters(), lr=1e-2, weight_decay=5e-4)
    best = 0
    best_state = None
    wait = 0
    for epoch in range(1, 200):
        m.train()
        opt.zero_grad()
        lg = m(feats)
        loss = F.cross_entropy(lg[idx_tr], labels[idx_tr])
        loss.backward()
        opt.step()
        m.eval()
        with torch.no_grad():
            lg = m(feats)
            va = accuracy(lg[idx_va], labels[idx_va])
        if va > best:
            best = va
            best_state = m.state_dict()
            wait = 0
        else:
            wait += 1
        if wait >= 30:
            break
    m.load_state_dict(best_state)
    with torch.no_grad():
        van = m(feats)
        v_acc = accuracy(van[idx_te], labels[idx_te])
        np.savez_compressed(van_path, logits=van.detach().numpy())
        print(f"vanilla {v_acc:.4f}")

train vanilla 150ep
vanilla 0.6801
